# NCBI Virus Dataset Creation

Author: Alexander Maksiaev

Purpose: Create dataset using NCBI and relabeling sequences. 

In [1]:
# Housekeeping

import os
import pandas as pd
import dateutil
import re
import shutil 
import numpy as np
import importlib
import utils  
importlib.reload(utils)
from utils import * 

In [3]:
# Paths and input

# Input
browser = input("Browser (Firefox, Chrome, or Edge): ")
sleep_time = input("Seconds to wait in between clicks (recommended 3): ")
# locations = input("Locations (separate with commas and no spaces in between locations): ")
# start_date = input("Start date (format: MM-DD-YYYY): ")
# end_date = input("End date: (format: MM-DD-YYYY): ")

# Dates and locations
start_date = "11-01-2021"
end_date = "11-14-2025"
date_range = start_date + "--" + end_date
locations = "Antarctica,North America,South America"

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# downloads = "C:/Users/maksi/Downloads/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder. Clean out downloads folder after each use. 
downloads_saved = home + "NCBI_Virus/downloads/" + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/" # "_Antarctica_North_America_South_America/"
andersen = home + "Andersen/avian-influenza/metadata/"
temp_files = home + "NCBI_Virus/temp/"
complete_files = home + "NCBI_Virus/complete/" + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it

references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"
# references = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/references/"
os.chdir(references)
states_ref = pd.read_csv("states_ref.csv")

# genotypes_df = pd.read_excel("genotype_key.xlsx")
# genotypes = list_df["Genotype"])

# serotype = "H5N1"
genotypes = ["B3.13", "D1.1", "D1.3"]
# genotypes = ["B3.2", "B3.6", "B3.7", "B3.5", "A3", "B3.13", "D1.1", "D1.3"]


## Downloading Data

In [4]:


# browser = "Chrome"
# sleep_time = "3"
# locations = "South America"
# start_date = "07-01-2025"
# end_date = "07-25-2025"



In [5]:
os.chdir(downloads)

if not os.path.exists(downloads_saved): # checking if the directory exists or not
    os.makedirs(downloads_saved) # if the directory is not present then create it

    # Move downloaded files to saved downloads
    for dirpath, dirs, files in os.walk(downloads):
        if len(files) > 0: # If we have any files that need to be moved
            for file in files:
                file_name = os.path.join(dirpath, file)
                destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
                try:
                    shutil.move(file_name, destination_path)
                except:
                    print("Error moving file", file_name)
                    continue 
        else: # If we don't have any downloaded files
            # Get files
            open_ncbi_virus(browser, sleep_time, locations, start_date, end_date)

            # Re-try 
            for dirpath, dirs, files in os.walk(downloads):
                if len(files) > 0: # If we have any files that need to be moved
                    for file in files:
                        file_name = os.path.join(dirpath, file)
                        destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
                        try:
                            shutil.move(file_name, destination_path)
                        except:
                            print("Error moving file", file_name)
                            continue 
                break 
        break 

## De-Duplication

In [6]:
os.chdir(downloads_saved)
metadata = pd.read_csv("sequences.csv")
print(len(metadata))

# Make sure we only have completed sequences -- 8 segments each 

metadata_counts = metadata.groupby(metadata.Isolate, as_index=False).size()
# print(metadata_counts)
metadata_counted = metadata.merge(metadata_counts, on="Isolate")

# Only keep those with size >= 8

metadata_complete_segs = metadata_counted[metadata_counted["size"] >= 8] # May have duplicates
metadata_complete_segs = metadata_complete_segs.drop_duplicates(subset="GenBank_Title", keep="last") # Get rid of duplicate segments

# Now only accept == 8 segments

metadata_segments = metadata_complete_segs[metadata_complete_segs["size"] == 8]
metadata_segments.head()

117090


,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Host,Tissue_Specimen_Source,Submitters,Organization,Org_location,Publications,Collection_Date,Release_Date,Molecule_type,size
0,PX508201.1,GenBank,GCA_053491305.1,SRR36016324,NaN,PRJNA1363857,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Homo sapiens,NaN,"Goya,S., Nunley,E., Greninger,A.L.","Greninger, Virology",USA,NaN,2025-11-10,2025-11-14,ssRNA(-),8
1,PX508202.1,GenBank,GCA_053491305.1,SRR36016324,NaN,PRJNA1363857,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Homo sapiens,NaN,"Goya,S., Nunley,E., Greninger,A.L.","Greninger, Virology",USA,NaN,2025-11-10,2025-11-14,ssRNA(-),8
2,PX508203.1,GenBank,GCA_053491305.1,SRR36016324,NaN,PRJNA1363857,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Homo sapiens,NaN,"Goya,S., Nunley,E., Greninger,A.L.","Greninger, Virology",USA,NaN,2025-11-10,2025-11-14,ssRNA(-),8
3,PX508204.1,GenBank,GCA_053491305.1,SRR36016324,NaN,PRJNA1363857,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Homo sapiens,NaN,"Goya,S., Nunley,E., Greninger,A.L.","Greninger, Virology",USA,NaN,2025-11-10,2025-11-14,ssRNA(-),8
4,PX508205.1,GenBank,GCA_053491305.1,SRR36016324,NaN,PRJNA1363857,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Homo sapiens,NaN,"Goya,S., Nunley,E., Greninger,A.L.","Greninger, Virology",USA,NaN,2025-11-10,2025-11-14,ssRNA(-),8


In [7]:
# # De-duplicate from Andersen using SRA Accession

# # If even one SRA Accession in this list exists in the NCBI Virus dataframe, remove it from NCBI Virus dataframe
# andersen_sras = []
# # Grab files
# for dirpath, dirs, files in os.walk(andersen):
#     for file in files:
#         file_name = os.path.join(dirpath, file)
#         if ".fasta" in file_name:
#             fasta_file = fasta_df_complete(file_name, states_ref) # Convert fasta file to dataframe
#             sra_accessions = fasta_file["Identifier"]
#             for value in sra_accessions.values:
#                 if "SRR" in value:
#                     andersen_sras.append(value)
#             isolates = fasta_file["Isolate_Id"]
#             for value in isolates.values:
#                 andersen_sras.append(value)
#             partials = fasta_file["Partials"]
#             for value in partials.values:
#                 andersen_sras.append(value)
#     break 

# # Grab more files
# # for dirpath, dirs, files in os.walk(andersen_dedup):
# #     for file in files:
# #         file_name = os.path.join(dirpath, file)
# #         if ".fasta" in file_name:
# #             fasta_file = fasta_df_complete(file_name, states_ref) # Convert fasta file to dataframe
# #             sra_accessions = fasta_file["Identifier"]
# #             for value in sra_accessions.values:
# #                 if "SRR" in value:
# #                     andersen_sras.append(value)
            
# metadata_segments["Partials"] = metadata_segments["Isolate"].apply(lambda x: partial_isolate(x) if x == x else x)
# # Remove duplicates from Andersen
# for value in andersen_sras: # to remove
#     if "SRR" in value:
#         metadata_segments = metadata_segments[metadata_segments["SRA_Accession"] != value]
    
#     metadata_segments = metadata_segments[metadata_segments["Isolate"] != value]
#     metadata_segments = metadata_segments[metadata_segments["Partials"] != value]
#     # print(value)
    
# print(len(metadata_segments))
# # metadata_segments
# # print(count)

## Add sequences to dataframe

In [8]:
# NCBI Virus Naming Convention Example:
# Influenza A virus |USA: IN|25-006338-001-original|H5N1|2025-02-20|Meleagris gallopavo|GenBank|SRR33124721|SAMN47941411|PRJNA980729|Influenza A virus (A/Turkey/IN/25-006338-001-original/2025(H5N1)) segment 1 polymerase PB2 (PB2) gene, complete cds
# Organism_Name | Geo_Location | Isolate | Genotype | Collection_Date | Host | GenBank_RefSeq | SRA_Accession | BioSample | BioProject | GenBank_Title | Accession

# Get sequences and headers together
# headers = []
# isolates = []
# sras = []
# headers_seqs = {}

os.chdir(downloads_saved)

sequences_fasta = df_from_fasta("sequences.fasta") # Turn fasta into a dataframe

# Filter dataframe to only include filtered SRA accessions
# sequences_fasta = sequences_fasta[sequences_fasta["full_header"].str.contains("|".join(list(metadata_segments["SRA_Accession"].values)))]
# sequences_fasta["SRA_Accession"] = sequences_fasta["full_header"].apply(lambda x: x.split("|")[-4])
sequences_fasta["Accession"] = sequences_fasta["full_header"].apply(lambda x: x.split("|")[-1][:-1])

# Extract segment number so that we can add the correct sequences to the correct sample
sequences_fasta["Segment"] = sequences_fasta["full_header"].apply(lambda x: int(re.search(r'segment (.?) ', x.split("|")[-2]).group(1)))

# Double-check the de-duplication
print(len(sequences_fasta)) 
print(sequences_fasta.head())
print(len(metadata_segments))

# Add sequences to the dataframe
metadata_segments = pd.merge(metadata_segments, sequences_fasta, on=["Accession", "Segment"])

117090
                                         full_header  \
0  >Influenza A virus |USA: Washington|2148|H5N5|...   
1  >Influenza A virus |USA: Washington|2148|H5N5|...   
2  >Influenza A virus |USA: Washington|2148|H5N5|...   
3  >Influenza A virus |USA: Washington|2148|H5N5|...   
4  >Influenza A virus |USA: Washington|2148|H5N5|...   

                                            sequence   Accession  Segment  
0  GGTTCAATCTGTCAAAATGGAGAACATAGTACTTCTTCTTGCAACA...  PX508201.1        4  
1  TAGATATTGAAAGATGAGTCTTCTAACCGAGGTCGAAACGTACGTT...  PX508202.1        7  
2  GTAGATAATCACTCACTGAGTGACATCAACATCATGGCGTCTCAAG...  PX508203.1        5  
3  GTGACAAAAACATAATGGATTTCAACACAGTGTCAAGCTTCCAGGT...  PX508204.1        8  
4  TACTGATCCAAAATGGAAGACTTTGTGCGACAATGCTTCAATCCAA...  PX508205.1        3  
100808


In [9]:
# # Make sure H5N1 is the only serotype we have

# metadata_segments["serotype"] = metadata_segments["full_header"].apply(lambda x: re.search(r'H.N.', x).group(0))

# metadata_segments = metadata_segments[metadata_segments["serotype"] == serotype]

## Find genotype using old genoflu results or Andersen Lab genoflu output

In [10]:
''' Add old genoflu results manually from previous folder to this one (downloads_saved), named "output_old.tsv" '''

' Add old genoflu results manually from previous folder to this one (downloads_saved), named "output_old.tsv" '

In [11]:
metadata_segments

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Submitters,Organization,Org_location,Publications,Collection_Date,Release_Date,Molecule_type,size,full_header,sequence
0,PX508201.1,GenBank,GCA_053491305.1,SRR36016324,NaN,PRJNA1363857,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Goya,S., Nunley,E., Greninger,A.L.","Greninger, Virology",USA,NaN,2025-11-10,2025-11-14,ssRNA(-),8,>Influenza A virus |USA: Washington|2148|H5N5|...,GGTTCAATCTGTCAAAATGGAGAACATAGTACTTCTTCTTGCAACA...
1,PX508202.1,GenBank,GCA_053491305.1,SRR36016324,NaN,PRJNA1363857,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Goya,S., Nunley,E., Greninger,A.L.","Greninger, Virology",USA,NaN,2025-11-10,2025-11-14,ssRNA(-),8,>Influenza A virus |USA: Washington|2148|H5N5|...,TAGATATTGAAAGATGAGTCTTCTAACCGAGGTCGAAACGTACGTT...
2,PX508203.1,GenBank,GCA_053491305.1,SRR36016324,NaN,PRJNA1363857,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Goya,S., Nunley,E., Greninger,A.L.","Greninger, Virology",USA,NaN,2025-11-10,2025-11-14,ssRNA(-),8,>Influenza A virus |USA: Washington|2148|H5N5|...,GTAGATAATCACTCACTGAGTGACATCAACATCATGGCGTCTCAAG...
3,PX508204.1,GenBank,GCA_053491305.1,SRR36016324,NaN,PRJNA1363857,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Goya,S., Nunley,E., Greninger,A.L.","Greninger, Virology",USA,NaN,2025-11-10,2025-11-14,ssRNA(-),8,>Influenza A virus |USA: Washington|2148|H5N5|...,GTGACAAAAACATAATGGATTTCAACACAGTGTCAAGCTTCCAGGT...
4,PX508205.1,GenBank,GCA_053491305.1,SRR36016324,NaN,PRJNA1363857,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Goya,S., Nunley,E., Greninger,A.L.","Greninger, Virology",USA,NaN,2025-11-10,2025-11-14,ssRNA(-),8,>Influenza A virus |USA: Washington|2148|H5N5|...,TACTGATCCAAAATGGAAGACTTTGTGCGACAATGCTTCAATCCAA...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100803,OK205883.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGGGTATCAGATATCAAAATGGAAAGAATAGTGATT...
100804,OK205884.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGTTAGATAATCACTCACCGAGTGACATTCACATCA...
100805,OK205885.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGAGTGAAGATGAATCCAAATCAGAAGATAATAACA...
100806,OK205886.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGTAGATATTGAAAGATGAGTCTTCTAACCGAGGTC...


In [12]:
genoflu_old = pd.read_csv("output_old.tsv", delimiter="\t")
genoflu_old = genoflu_old.rename(columns={"Strain":"Accession_Root"})
metadata_segments["Accession_Root"] = metadata_segments["Accession"].values[0].split(".")[0]

metadata_segments_old = metadata_segments.merge(genoflu_old, how="inner", on="Accession_Root")
metadata_segments_new = metadata_segments.merge(genoflu_old,indicator = True, how='left').loc[lambda x : x['_merge']!='both']

print(len(metadata_segments_old))
print(len(metadata_segments_new))

0
100808


In [13]:
os.chdir(andersen)

genoflu_andersen = pd.read_csv("genoflu_results.tsv", delimiter="\t")
genoflu_andersen = genoflu_andersen.rename(columns={"sample":"SRA_Accession"})

metadata_segments_known = metadata_segments.merge(genoflu_andersen, how="inner", on="SRA_Accession")

metadata_segments_unknown = metadata_segments.merge(genoflu_andersen, indicator = True, how='left', on="SRA_Accession").loc[lambda x : x['_merge']!='both']

print(genoflu_andersen)

      SRA_Accession                 date       File Name  \
0       SRR30789580  2025-05-09_10-49-10  SRR30789580.fa   
1       SRR32973809  2025-05-09_10-52-30  SRR32973809.fa   
2       SRR32125669  2025-05-09_10-47-22  SRR32125669.fa   
3       SRR31597237  2025-05-09_10-47-24  SRR31597237.fa   
4       SRR31605135  2025-05-09_10-47-38  SRR31605135.fa   
...             ...                  ...             ...   
11680   SRR35950025  2025-11-07_05-15-50  SRR35950025.fa   
11681   SRR35950026  2025-11-07_05-15-51  SRR35950026.fa   
11682   SRR35950027  2025-11-07_05-15-50  SRR35950027.fa   
11683   SRR35950028  2025-11-07_05-15-50  SRR35950028.fa   
11684   SRR35950029  2025-11-07_05-15-50  SRR35950029.fa   

                                                Genotype  \
0      Not assigned: Only 0 segments >98.0% match fou...   
1                                                   D1.3   
2                                                  B3.13   
3                                      

In [14]:
metadata_segments_known

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,sequence,Accession_Root,date,File Name,Genotype_y,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List
0,PX440412.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,ATGGAGAGAATAAAAGAACTGAGAGATCTAATGTCACAGTCTCGCA...,PX508201,2025-08-29_06-15-55,SRR35150543.fa,B3.13,"NA:ea1, MP:ea1, HA:ea1, PB2:am2.2, NS:am1.1, N...","ea1:22-003707-003:NA, ea1:22-003707-003:MP, ea...","98.65%, 98.98%, 98.18%, 98.42%, 98.81%, 98.40%...","19, 10, 31, 36, 10, 24, 27, 21",Ran on FASTA - No Coverage Report
1,PX440413.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,ATGGATGTCAATCCGACCTTACTCTTCTTGAAAGTTCCAGCGCAAA...,PX508201,2025-08-29_06-15-55,SRR35150543.fa,B3.13,"NA:ea1, MP:ea1, HA:ea1, PB2:am2.2, NS:am1.1, N...","ea1:22-003707-003:NA, ea1:22-003707-003:MP, ea...","98.65%, 98.98%, 98.18%, 98.42%, 98.81%, 98.40%...","19, 10, 31, 36, 10, 24, 27, 21",Ran on FASTA - No Coverage Report
2,PX440414.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,ATGGAAGACTTTGTGCGACAATGCTTCAATCCAATGATTGTCGAGC...,PX508201,2025-08-29_06-15-55,SRR35150543.fa,B3.13,"NA:ea1, MP:ea1, HA:ea1, PB2:am2.2, NS:am1.1, N...","ea1:22-003707-003:NA, ea1:22-003707-003:MP, ea...","98.65%, 98.98%, 98.18%, 98.42%, 98.81%, 98.40%...","19, 10, 31, 36, 10, 24, 27, 21",Ran on FASTA - No Coverage Report
3,PX440415.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,ATGAAGAACATAGTACTACTTCTTGCAATAGTTAGCCTTGTTAAAA...,PX508201,2025-08-29_06-15-55,SRR35150543.fa,B3.13,"NA:ea1, MP:ea1, HA:ea1, PB2:am2.2, NS:am1.1, N...","ea1:22-003707-003:NA, ea1:22-003707-003:MP, ea...","98.65%, 98.98%, 98.18%, 98.42%, 98.81%, 98.40%...","19, 10, 31, 36, 10, 24, 27, 21",Ran on FASTA - No Coverage Report
4,PX440416.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAAATGGAAACTG...,PX508201,2025-08-29_06-15-55,SRR35150543.fa,B3.13,"NA:ea1, MP:ea1, HA:ea1, PB2:am2.2, NS:am1.1, N...","ea1:22-003707-003:NA, ea1:22-003707-003:MP, ea...","98.65%, 98.98%, 98.18%, 98.42%, 98.81%, 98.40%...","19, 10, 31, 36, 10, 24, 27, 21",Ran on FASTA - No Coverage Report
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38235,PQ012131.1,GenBank,GCA_040780295.1,SRR29281472,SAMN41656639,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,ATGGAGAACATAGTACTACTTCTTGCAATAGTTAGCCTTGTTAAAA...,PX508201,2025-05-09_10-47-50,SRR29281472.fa,B3.13,"NP:am8, NA:ea1, NS:am1.1, PB2:am2.2, MP:ea1, H...","am8:23-032005-001:NP, ea1:22-003707-003:NA, am...","99.13%, 98.94%, 99.28%, 98.73%, 98.88%, 98.71%...","13, 15, 6, 29, 11, 22, 22, 10",Ran on FASTA - No Coverage Report
38236,PQ012132.1,GenBank,GCA_040780295.1,SRR29281472,SAMN41656639,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAAATGGAAACTG...,PX508201,2025-05-09_10-47-50,SRR29281472.fa,B3.13,"NP:am8, NA:ea1, NS:am1.1, PB2:am2.2, MP:ea1, H...","am8:23-032005-001:NP, ea1:22-003707-003:NA, am...","99.13%, 98.94%, 99.28%, 98.73%, 98.88%, 98.71%...","13, 15, 6, 29, 11, 22, 22, 10",Ran on FASTA - No Coverage Report
38237,PQ012133.1,GenBank,GCA_040780295.1,SRR29281472,SAMN41656639,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,ATGAATCCAAATCA

In [15]:
metadata_segments_unknown

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Accession_Root,date,File Name,Genotype_y,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List,_merge
0,PX508201.1,GenBank,GCA_053491305.1,SRR36016324,NaN,PRJNA1363857,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,PX508201,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1,PX508202.1,GenBank,GCA_053491305.1,SRR36016324,NaN,PRJNA1363857,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,PX508201,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2,PX508203.1,GenBank,GCA_053491305.1,SRR36016324,NaN,PRJNA1363857,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,PX508201,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3,PX508204.1,GenBank,GCA_053491305.1,SRR36016324,NaN,PRJNA1363857,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,PX508201,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4,PX508205.1,GenBank,GCA_053491305.1,SRR36016324,NaN,PRJNA1363857,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,PX508201,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100803,OK205883.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,PX508201,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
100804,OK205884.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,PX508201,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
100805,OK205885.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,PX508201,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
100806,OK205886.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,PX508201,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


## Create FASTA files of unknown genotypes using deduplicated sequences

In [16]:
# # Create 1 fasta file per header
# metadata_segments_unknown["Partial_Header"] = metadata_segments_unknown["full_header"].apply(lambda x: "|".join(x.split("|")[:-2]))

# # Create list of dataframes
# df_list = []
# for partial_header in list(set(metadata_segments_unknown["Partial_Header"].values)): # Unique partial headers only
#     # Get smaller dataframe
#     df = metadata_segments_unknown[metadata_segments_unknown["Partial_Header"] == partial_header]
#     df = df.sort_values(by="Segment")
#     # Make sure there are 8 segments
#     if len(df) == 8:
#         # Forbidden characters in headers
#         for c in ["/", "|", " ", ":", ",", "(", ")", "-"]:
#             df["full_header"] = df["full_header"].apply(lambda x: x.replace(c, "_"))
#             df_list.append(df)

# # Make fasta files
# for df in df_list:
#     # Forbidden characters in file name 
#     title = df["Accession"].values[0].split(".")[0]
#     # for c in [">", "/", "|", " ", ":", ",", "(", ")", "-"]:
#     #     title = title.replace(c, "_")
#     df_to_fasta(df, title + ".fasta", temp_files)

In [17]:
# Create 8 fasta files per segment

# Get all the segments
metadata_segments_unknown["Partial_Header"] = metadata_segments_unknown["full_header"].apply(lambda x: "|".join(x.split("|")[:-2]))

# Create list of dataframes
df_list = []
for partial_header in list(set(metadata_segments_unknown["Partial_Header"].values)): # Unique partial headers only
    # Get smaller dataframe
    df = metadata_segments_unknown[metadata_segments_unknown["Partial_Header"] == partial_header]
    df = df.sort_values(by="Segment")
    # print(partial_header)
    # print(df["Segment"])
    # break 
    # Make sure there are 8 segments
    if len(df) == 8:
        # Forbidden characters in headers
        for c in ["/", "|", " ", ":", ",", "(", ")", "-"]:
            df["Partial_Header"] = df["Partial_Header"].apply(lambda x: x.replace(c, "_"))
            # if c == "-":
        df["full_header"] = df["Partial_Header"]
        df_list.append(df)

print(df_list[0]["full_header"])

# Make fasta files
for df in df_list:
    # Forbidden characters in file name 
    # title = df["Accession"].values[0].split(".")[0]
    segments = list(set(df["Segment"].apply(lambda x: int(x)).values))
    # print(df)
    # print(segments)
    # break 
    # for c in [">", "/", "|", " ", ":", ",", "(", ")", "-"]:
    #     title = title.replace(c, "_")
    for segment in segments:
        one_row = df[df["Segment"] == segment]
        # print(one_row["full_header"])
        # print(segment)
        # break 
        df_to_fasta(one_row, str(segment) + "_seg.fasta", temp_files)
    # break 

29856    >Influenza_A_virus__Canada__SK_FAV_1386_1_2022...
29857    >Influenza_A_virus__Canada__SK_FAV_1386_1_2022...
29858    >Influenza_A_virus__Canada__SK_FAV_1386_1_2022...
29859    >Influenza_A_virus__Canada__SK_FAV_1386_1_2022...
29860    >Influenza_A_virus__Canada__SK_FAV_1386_1_2022...
29861    >Influenza_A_virus__Canada__SK_FAV_1386_1_2022...
29862    >Influenza_A_virus__Canada__SK_FAV_1386_1_2022...
29863    >Influenza_A_virus__Canada__SK_FAV_1386_1_2022...
Name: full_header, dtype: object


## Re-Labeling Using GenoFlu

**STOP HERE AND USE GENOFLU TO FIND NEW GENOTYPES.** Then, make sure "output.tsv" is in the downloads directory.



In [ ]:
'''
To run GenoFLU-multi, first change directories (and activate genoflu conda environment):

conda activate genoflu
cd GenoFLU-multi

And then call the python script:

python bin/genoflu-multi.py -f <FASTA_directory>
'''

'\n## In Ubuntu 22.04.3 LTS ##\n\nconda activate genoflu\n\n## Genoflu.py iteratively through all FASTAs in directory ##\n\nfor file in *.fasta; do genoflu.py -f "$file"; sleep 0.75; done\n\n## Concatenate output files ##\n\n awk \'(NR == 1) || (FNR > 1)\' *.tsv > output.tsv\n'

In [18]:
metadata_segments_known

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,serotype,Accession_Root,date,File Name,Genotype_y,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List
0,PX440412.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,H5N1,PX440412,2025-08-29_06-15-55,SRR35150543.fa,B3.13,"NA:ea1, MP:ea1, HA:ea1, PB2:am2.2, NS:am1.1, N...","ea1:22-003707-003:NA, ea1:22-003707-003:MP, ea...","98.65%, 98.98%, 98.18%, 98.42%, 98.81%, 98.40%...","19, 10, 31, 36, 10, 24, 27, 21",Ran on FASTA - No Coverage Report
1,PX440413.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,H5N1,PX440412,2025-08-29_06-15-55,SRR35150543.fa,B3.13,"NA:ea1, MP:ea1, HA:ea1, PB2:am2.2, NS:am1.1, N...","ea1:22-003707-003:NA, ea1:22-003707-003:MP, ea...","98.65%, 98.98%, 98.18%, 98.42%, 98.81%, 98.40%...","19, 10, 31, 36, 10, 24, 27, 21",Ran on FASTA - No Coverage Report
2,PX440414.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,H5N1,PX440412,2025-08-29_06-15-55,SRR35150543.fa,B3.13,"NA:ea1, MP:ea1, HA:ea1, PB2:am2.2, NS:am1.1, N...","ea1:22-003707-003:NA, ea1:22-003707-003:MP, ea...","98.65%, 98.98%, 98.18%, 98.42%, 98.81%, 98.40%...","19, 10, 31, 36, 10, 24, 27, 21",Ran on FASTA - No Coverage Report
3,PX440415.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,H5N1,PX440412,2025-08-29_06-15-55,SRR35150543.fa,B3.13,"NA:ea1, MP:ea1, HA:ea1, PB2:am2.2, NS:am1.1, N...","ea1:22-003707-003:NA, ea1:22-003707-003:MP, ea...","98.65%, 98.98%, 98.18%, 98.42%, 98.81%, 98.40%...","19, 10, 31, 36, 10, 24, 27, 21",Ran on FASTA - No Coverage Report
4,PX440416.1,GenBank,GCA_052945965.1,SRR35150543,SAMN50804733,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,H5N1,PX440412,2025-08-29_06-15-55,SRR35150543.fa,B3.13,"NA:ea1, MP:ea1, HA:ea1, PB2:am2.2, NS:am1.1, N...","ea1:22-003707-003:NA, ea1:22-003707-003:MP, ea...","98.65%, 98.98%, 98.18%, 98.42%, 98.81%, 98.40%...","19, 10, 31, 36, 10, 24, 27, 21",Ran on FASTA - No Coverage Report
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38179,PQ012131.1,GenBank,GCA_040780295.1,SRR29281472,SAMN41656639,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,H5N1,PX440412,2025-05-09_10-47-50,SRR29281472.fa,B3.13,"NP:am8, NA:ea1, NS:am1.1, PB2:am2.2, MP:ea1, H...","am8:23-032005-001:NP, ea1:22-003707-003:NA, am...","99.13%, 98.94%, 99.28%, 98.73%, 98.88%, 98.71%...","13, 15, 6, 29, 11, 22, 22, 10",Ran on FASTA - No Coverage Report
38180,PQ012132.1,GenBank,GCA_040780295.1,SRR29281472,SAMN41656639,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,H5N1,PX440412,2025-05-09_10-47-50,SRR29281472.fa,B3.13,"NP:am8, NA:ea1, NS:am1.1, PB2:am2.2, MP:ea1, H...","am8:23-032005-001:NP, ea1:22-003707-003:NA, am...","99.13%, 98.94%, 99.28%, 98.73%, 98.88%, 98.71%...","13, 15, 6, 29, 11, 22, 22, 10",Ran on FASTA - No Coverage Report
38181,PQ012133.1,GenBank,GCA_040780295.1,SRR29281472,SAMN41656639,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,H5N1,PX440412,2025-05-09_10-47-50,SRR29281472.fa,B3.13,"NP:am8, NA:ea1, NS:am1.1, PB2:am2.2, MP:ea1, H...","am8:23-032005-001:NP, ea1:22-003707-003:NA, am...","99.13%, 98.94%, 99.28%, 98.73%, 98.88%, 98.71%...","13, 15, 6, 29, 11, 22, 22, 10",Ran on FASTA - No Coverage Report
38182,PQ012134.1,GenBank,GCA_040780295.1,SRR29281472

In [32]:
# Merging

os.chdir(downloads_saved)

# Read in genoflu results
output_genoflu = pd.read_csv("output.tsv", delimiter="\t")

metadata_segments_unknown["Partial_Header"] = metadata_segments_unknown["full_header"].apply(lambda x: "|".join(x.split("|")[:-2]))
for c in ["/", "|", " ", ":", ",", "(", ")", "-"]:
    metadata_segments_unknown["Partial_Header"] = metadata_segments_unknown["Partial_Header"].apply(lambda x: x.replace(c, "_"))

metadata_segments_unknown["Partial_Header"] = metadata_segments_unknown["Partial_Header"].apply(lambda x: x.replace(">", ""))

metadata_segments_unknown["Strain"] = metadata_segments_unknown["Partial_Header"]

metadata_genoflu = metadata_segments_unknown.merge(output_genoflu, how="left", on="Strain") 

# Fill the rest of the 8 segments with the same genotype
metadata_genoflu = metadata_genoflu.ffill()

print(metadata_genoflu)

print(metadata_genoflu["Genotype_x"])

print(metadata_genoflu.columns)


        Accession GenBank_RefSeq         Assembly SRA_Accession     BioSample  \
0      PX508201.1        GenBank  GCA_053491305.1   SRR36016324           NaN   
1      PX508202.1        GenBank  GCA_053491305.1   SRR36016324           NaN   
2      PX508203.1        GenBank  GCA_053491305.1   SRR36016324           NaN   
3      PX508204.1        GenBank  GCA_053491305.1   SRR36016324           NaN   
4      PX508205.1        GenBank  GCA_053491305.1   SRR36016324           NaN   
...           ...            ...              ...           ...           ...   
62563  OK205883.1        GenBank  GCA_038165665.1   SRR23852495  SAMN33745292   
62564  OK205884.1        GenBank  GCA_038165665.1   SRR23852495  SAMN33745292   
62565  OK205885.1        GenBank  GCA_038165665.1   SRR23852495  SAMN33745292   
62566  OK205886.1        GenBank  GCA_038165665.1   SRR23852495  SAMN33745292   
62567  OK205887.1        GenBank  GCA_038165665.1   SRR23852495  SAMN33745292   

         BioProject      Or

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_16096\1490668429.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  metadata_genoflu = metadata_genoflu.ffill()


# Concatenate with known genotypes

In [33]:
metadata_segments_known["Genotype"] = metadata_segments_known["Genotype_y"]
metadata_segments_known["Serotype"] = metadata_segments_known["Genotype_x"]

metadata_genoflu = pd.concat([metadata_genoflu, metadata_segments_known])

metadata_genoflu.columns

Index(['Accession', 'GenBank_RefSeq', 'Assembly', 'SRA_Accession', 'BioSample',
       'BioProject', 'Organism_Name', 'Species', 'Genus', 'Family',
       'Genotype_x', 'Isolate', 'Segment', 'GenBank_Title', 'Length',
       'Nuc_Completeness', 'Geo_Location', 'Country', 'USA', 'Host',
       'Tissue_Specimen_Source', 'Submitters', 'Organization', 'Org_location',
       'Publications', 'Collection_Date', 'Release_Date', 'Molecule_type',
       'size', 'full_header', 'sequence', 'Accession_Root', 'date',
       'File Name', 'Genotype_y', 'Genotype List Used, >=98.0%_x',
       'Genotype Sample Title List_x', 'Genotype Percent Match List_x',
       'Genotype Mismatch List_x', 'Genotype Average Depth of Coverage List_x',
       '_merge', 'Partial_Header', 'Strain', 'Genotype',
       'Genotype List Used, >=98.0%_y', 'Genotype Sample Title List_y',
       'Genotype Percent Match List_y', 'Genotype Mismatch List_y',
       'Genotype Average Depth of Coverage List_y', 'Date run',
       'Gen

In [37]:
metadata_genoflu = metadata_genoflu[["Accession", "GenBank_Title", "Host", "Collection_Date", "SRA_Accession", "Isolate", "Genotype", "Geo_Location", "full_header", "sequence", "Serotype", "Segment", "File Name", "Partial_Header", "Strain"]]

metadata_genoflu["genbank_name"] = metadata_genoflu["GenBank_Title"].apply(lambda x: x.split("(")[1] )
metadata_genoflu["Host"] = metadata_genoflu["genbank_name"].apply(lambda x: x.split("/")[1])

metadata_genoflu = metadata_genoflu.dropna(subset="SRA_Accession")
# metadata_genoflu = metadata_genoflu[~metadata_genoflu.duplicated(["Segment", "SRA_Accession"], keep=False).groupby(df["SRA_Accession"]).transform('sum').ge(8)]
# metadata_genoflu = metadata_genoflu.drop_duplicates(subset=["SRA_Accession", "Segment"], keep="first", inplace=True)

metadata_genoflu # [metadata_genoflu["Genotype"]  == "B3.13"]

,Accession,GenBank_Title,Host,Collection_Date,SRA_Accession,Isolate,Genotype,Geo_Location,full_header,sequence,Serotype,Segment,File Name,Partial_Header,Strain,genbank_name
0,PX508201.1,Influenza A virus (A/Washington/2148/2025(H5N5...,Washington,2025-11-10,SRR36016324,2148,A6,USA: Washington,>Influenza A virus |USA: Washington|2148|H5N5|...,GGTTCAATCTGTCAAAATGGAGAACATAGTACTTCTTCTTGCAACA...,NaN,4,NaN,Influenza_A_virus__USA__Washington_2148_H5N5_2...,Influenza_A_virus__USA__Washington_2148_H5N5_2...,A/Washington/2148/2025
1,PX508202.1,Influenza A virus (A/Washington/2148/2025(H5N5...,Washington,2025-11-10,SRR36016324,2148,A6,USA: Washington,>Influenza A virus |USA: Washington|2148|H5N5|...,TAGATATTGAAAGATGAGTCTTCTAACCGAGGTCGAAACGTACGTT...,NaN,7,NaN,Influenza_A_virus__USA__Washington_2148_H5N5_2...,Influenza_A_virus__USA__Washington_2148_H5N5_2...,A/Washington/2148/2025
2,PX508203.1,Influenza A virus (A/Washington/2148/2025(H5N5...,Washington,2025-11-10,SRR36016324,2148,A6,USA: Washington,>Influenza A virus |USA: Washington|2148|H5N5|...,GTAGATAATCACTCACTGAGTGACATCAACATCATGGCGTCTCAAG...,NaN,5,NaN,Influenza_A_virus__USA__Washington_2148_H5N5_2...,Influenza_A_virus__USA__Washington_2148_H5N5_2...,A/Washington/2148/2025
3,PX508204.1,Influenza A virus (A/Washington/2148/2025(H5N5...,Washington,2025-11-10,SRR36016324,2148,A6,USA: Washington,>Influenza A virus |USA: Washington|2148|H5N5|...,GTGACAAAAACATAATGGATTTCAACACAGTGTCAAGCTTCCAGGT...,NaN,8,NaN,Influenza_A_virus__USA__Washington_2148_H5N5_2...,Influenza_A_virus__USA__Washington_2148_H5N5_2...,A/Washington/2148/2025
4,PX508205.1,Influenza A virus (A/Washington/2148/2025(H5N5...,Washington,2025-11-10,SRR36016324,2148,A6,USA: Washington,>Influenza A virus |USA: Washington|2148|H5N5|...,TACTGATCCAAAATGGAAGACTTTGTGCGACAATGCTTCAATCCAA...,NaN,3,NaN,Influenza_A_virus__USA__Washington_2148_H5N5_2...,Influenza_A_virus__USA__Washington_2148_H5N5_2...,A/Washington/2148/2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38235,PQ012131.1,Influenza A virus (A/Turkey/Minnesota/24-01468...,Turkey,2024-05-18,SRR29281472,24-014685-001-original,B3.13,USA:MN,>Influenza A virus |USA:MN|24-014685-001-origi...,ATGGAGAACATAGTACTACTTCTTGCAATAGTTAGCCTTGTTAAAA...,H5N1,4,SRR29281472.fa,NaN,NaN,A/Turkey/Minnesota/24-014685-001-original/2024
38236,PQ012132.1,Influenza A virus (A/Turkey/Minnesota/24-01468...,Turkey,2024-05-18,SRR29281472,24-014685-001-original,B3.13,USA:MN,>Influenza A virus |USA:MN|24-014685-001-origi...,ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAAATGGAAACTG...,H5N1,5,SRR29281472.fa,NaN,NaN,A/Turkey/Minnesota/24-014685-001-original/2024
38237,PQ012133.1,Influenza A virus (A/Turkey/Minnesota/24-01468...,Turkey,2024-05-18,SRR29281472,24-014685-001-original,B3.13,USA:MN,>Influenza A virus |USA:MN|24-014685-001-origi...,ATGAATCCAAATCAAAAGATAACAACCATTGGATCAATCTGTATGG...,H5N1,6,SRR29281472.fa,NaN,NaN,A/Turkey/Minnesota/24-014685-001-original/2024
38238,PQ012134.1,Influenza A virus (A/Turkey/Minnesota/24-01468...,Turkey,2024-05-18,SRR29281472,24-014685-001-original,B3.13,USA:MN,>Influenza A virus |USA:MN|24-014685-001-origi...,ATGAGTCTTCTAACCGAGGTCGAAACGTACGTTCTCTCTATCGTCC...,H5N1,7,SRR29281472.fa,NaN,NaN,A/Turkey/Minnesota/24-014685-001-original/2024


We want:
* Host
* Geo-Location
* Isolate
* Year
* Collection Date
* Host Type
* Genotype

In [39]:
# Animals 

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata_genoflu)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


['red-breasted merganser', 'mergus', 'common eider', 'eared grebe', 'white-winged scoter', 'pheasant', 'glaucous gull', 'long-eared owl', 'greater scaup', 'white-winged dove', 'otaria flavescens', 'crane', 'savannah cat', 'nevada', 'wyoming', 'nannopterum brasilianus', 'greater white-fronted goose', 'red-necked grebe', 'cattle milk product', 'snowy egret', 'duck', 'raccoon', 'rock pigeon', 'green heron', 'domestic cat', 'michigan', 'great-horned owl', 'bos taurus', 'antofagasta', 'bobcat', 'american crow', 'thayers gull', 'anatinae', 'lynx', 'dunlin', 'gallus', 'american goshawk', 'goose', 'hawk', 'emu', 'cat', 'gyrfalcon', 'wild bird', 'serval', 'thalasseus maximus', 'western sandpiper', 'common merganser', 'black-billed magpie', 'american black duck', 'guinea fowl', 'broad-winged hawk', 'gray seal', 'domestic turkey', 'atlantic puffin', 'california condor', 'surf scoter', 'heron', 'american kestrel', 'rosy-billed pochard', 'crested caracara', 'great egret', 'mottled duck', 'bald eagl

In [40]:
metadata_genoflu

,Accession,GenBank_Title,Host,Collection_Date,SRA_Accession,Isolate,Genotype,Geo_Location,full_header,sequence,Serotype,Segment,File Name,Partial_Header,Strain,genbank_name
0,PX508201.1,Influenza A virus (A/Washington/2148/2025(H5N5...,Washington,2025-11-10,SRR36016324,2148,A6,USA: Washington,>Influenza A virus |USA: Washington|2148|H5N5|...,GGTTCAATCTGTCAAAATGGAGAACATAGTACTTCTTCTTGCAACA...,NaN,4,NaN,Influenza_A_virus__USA__Washington_2148_H5N5_2...,Influenza_A_virus__USA__Washington_2148_H5N5_2...,A/Washington/2148/2025
1,PX508202.1,Influenza A virus (A/Washington/2148/2025(H5N5...,Washington,2025-11-10,SRR36016324,2148,A6,USA: Washington,>Influenza A virus |USA: Washington|2148|H5N5|...,TAGATATTGAAAGATGAGTCTTCTAACCGAGGTCGAAACGTACGTT...,NaN,7,NaN,Influenza_A_virus__USA__Washington_2148_H5N5_2...,Influenza_A_virus__USA__Washington_2148_H5N5_2...,A/Washington/2148/2025
2,PX508203.1,Influenza A virus (A/Washington/2148/2025(H5N5...,Washington,2025-11-10,SRR36016324,2148,A6,USA: Washington,>Influenza A virus |USA: Washington|2148|H5N5|...,GTAGATAATCACTCACTGAGTGACATCAACATCATGGCGTCTCAAG...,NaN,5,NaN,Influenza_A_virus__USA__Washington_2148_H5N5_2...,Influenza_A_virus__USA__Washington_2148_H5N5_2...,A/Washington/2148/2025
3,PX508204.1,Influenza A virus (A/Washington/2148/2025(H5N5...,Washington,2025-11-10,SRR36016324,2148,A6,USA: Washington,>Influenza A virus |USA: Washington|2148|H5N5|...,GTGACAAAAACATAATGGATTTCAACACAGTGTCAAGCTTCCAGGT...,NaN,8,NaN,Influenza_A_virus__USA__Washington_2148_H5N5_2...,Influenza_A_virus__USA__Washington_2148_H5N5_2...,A/Washington/2148/2025
4,PX508205.1,Influenza A virus (A/Washington/2148/2025(H5N5...,Washington,2025-11-10,SRR36016324,2148,A6,USA: Washington,>Influenza A virus |USA: Washington|2148|H5N5|...,TACTGATCCAAAATGGAAGACTTTGTGCGACAATGCTTCAATCCAA...,NaN,3,NaN,Influenza_A_virus__USA__Washington_2148_H5N5_2...,Influenza_A_virus__USA__Washington_2148_H5N5_2...,A/Washington/2148/2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38235,PQ012131.1,Influenza A virus (A/Turkey/Minnesota/24-01468...,Turkey,2024-05-18,SRR29281472,24-014685-001-original,B3.13,USA:MN,>Influenza A virus |USA:MN|24-014685-001-origi...,ATGGAGAACATAGTACTACTTCTTGCAATAGTTAGCCTTGTTAAAA...,H5N1,4,SRR29281472.fa,NaN,NaN,A/Turkey/Minnesota/24-014685-001-original/2024
38236,PQ012132.1,Influenza A virus (A/Turkey/Minnesota/24-01468...,Turkey,2024-05-18,SRR29281472,24-014685-001-original,B3.13,USA:MN,>Influenza A virus |USA:MN|24-014685-001-origi...,ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAAATGGAAACTG...,H5N1,5,SRR29281472.fa,NaN,NaN,A/Turkey/Minnesota/24-014685-001-original/2024
38237,PQ012133.1,Influenza A virus (A/Turkey/Minnesota/24-01468...,Turkey,2024-05-18,SRR29281472,24-014685-001-original,B3.13,USA:MN,>Influenza A virus |USA:MN|24-014685-001-origi...,ATGAATCCAAATCAAAAGATAACAACCATTGGATCAATCTGTATGG...,H5N1,6,SRR29281472.fa,NaN,NaN,A/Turkey/Minnesota/24-014685-001-original/2024
38238,PQ012134.1,Influenza A virus (A/Turkey/Minnesota/24-01468...,Turkey,2024-05-18,SRR29281472,24-014685-001-original,B3.13,USA:MN,>Influenza A virus |USA:MN|24-014685-001-origi...,ATGAGTCTTCTAACCGAGGTCGAAACGTACGTTCTCTCTATCGTCC...,H5N1,7,SRR29281472.fa,NaN,NaN,A/Turkey/Minnesota/24-014685-001-original/2024


In [42]:
# Re-Labeling

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Make sure NaN doesn't mess up the whole name
metadata_genoflu = metadata_genoflu.fillna("")
metadata_genoflu["Host"] = metadata_genoflu["Host"].apply(str.lower)

# Fix animals
fix_animals_andersen(metadata_genoflu, animals_ref)

# Get the years
metadata_genoflu["Years"] = metadata_genoflu["Collection_Date"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y"))

metadata_genoflu["Geo_Location_Abrv"] = metadata_genoflu["Geo_Location"].apply(lambda x: 
                                                                        # If "x" has the state abbreviation (e.g. "MD")
                                                                        states_ref.loc[states_ref["Abbreviation"].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0] 
                                                                        + "-" + 
                                                                        x.split(" ")[-1]
                                                                        if states_ref["Abbreviation"].str.contains("|".join((re.sub(":? ", ",", x).replace(" ", "_").split(','))), regex=True).any()
                                                                        # If "x" has the full state name (e.g. "Maryland")
                                                                        else states_ref.loc[states_ref['State'].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0]
                                                                        + "-" + 
                                                                        states_ref.loc[states_ref['State'].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Abbreviation'].iloc[0] 
                                                                        if states_ref["State"].str.contains("|".join((re.sub(":? ", ",", x).replace(" ", "_").split(','))), regex=True).any()
                                                                        # If "x" has neither the state abbreviation nor the full state name nor is "USA"
                                                                        else 
                                                                        "USA"
                                                                        )

metadata_genoflu["Geo_Location_Abrv"] = metadata_genoflu["Geo_Location_Abrv"].apply(lambda x: x.split("-")[0] if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] else x)


# Make new labels
names = ">" + metadata_genoflu["SRA_Accession"].apply(lambda x: x.split(",")[0]) + "|" + metadata_genoflu["genbank_name"] + "|" + metadata_genoflu["Serotype"] + "|" + metadata_genoflu["Geo_Location_Abrv"] + "|" + metadata_genoflu["Collection_Date"] + "|" + metadata_genoflu["Host_Type"] + "|" + metadata_genoflu["Genotype"]

metadata_genoflu["Name"] = names

print(metadata_genoflu["Geo_Location_Abrv"])


0        USA-WA
1        USA-WA
2        USA-WA
3        USA-WA
4        USA-WA
          ...  
38235       USA
38236       USA
38237       USA
38238       USA
38239       USA
Name: Geo_Location_Abrv, Length: 100808, dtype: object


## Rename segments and make complete FASTA files

In [50]:
# Set up segments

segments = {1:"PB2", 2:"PB1", 3:"PA", 4:"HA", 5:"NP", 6:"NA", 7:"MP", 8:"NS"}
metadata_genoflu["Segment_Name"] = metadata_genoflu["Segment"].apply(lambda x: int(x)).map(segments)

# Separate into several dataframes based on genotype + segment
segment_genotype_dfs = []
for segment in segments.values():
    m_g = metadata_genoflu[metadata_genoflu["Segment_Name"] == segment]
    for genotype in list(set(m_g["Genotype"].values)):
        if genotype in genotypes:
            df = m_g[(m_g["Genotype"] == genotype)] 
            df.drop_duplicates(subset="SRA_Accession", keep="first", inplace=True)
            segment_genotype_dfs.append(df)
            pair = genotype + "_" + segment
            print(pair)
        elif "Not assigned" in genotype:
            print(genotype)
            df = m_g[(m_g["Genotype"].str.contains("Not assigned"))] 
            df.drop_duplicates(subset="SRA_Accession", keep="first", inplace=True)
            segment_genotype_dfs.append(df)
            pair = "Unassigned_" + segment
            print(pair)

Not assigned: No Matching Genotypes
Unassigned_PB2
Not assigned: Only 7 segments >98.0% match found of total 8 segments in input file
Unassigned_PB2
Not assigned: Only 1 segments >98.0% match found of total 8 segments in input file
Unassigned_PB2
Not assigned: Only 4 segments >98.0% match found of total 8 segments in input file
Unassigned_PB2
Not assigned: Only 5 segments >98.0% match found of total 8 segments in input file
Unassigned_PB2
Not assigned: Only 2 segments >98.0% match found of total 8 segments in input file
Unassigned_PB2
D1.1_PB2
D1.3_PB2
B3.13_PB2
Not assigned: Only 0 segments >98.0% match found of total 8 segments in input file
Unassigned_PB2
Not assigned: Only 3 segments >98.0% match found of total 8 segments in input file
Unassigned_PB2
Not assigned: Only 6 segments >98.0% match found of total 8 segments in input file
Unassigned_PB2
Not assigned: No Matching Genotypes
Unassigned_PB1
Not assigned: Only 7 segments >98.0% match found of total 8 segments in input file
Una

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_16096\3807190993.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(subset="SRA_Accession", keep="first", inplace=True)
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_16096\3807190993.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(subset="SRA_Accession", keep="first", inplace=True)


Unassigned_PA
Not assigned: Only 7 segments >98.0% match found of total 8 segments in input file
Unassigned_PA
Not assigned: Only 1 segments >98.0% match found of total 8 segments in input file
Unassigned_PA
Not assigned: Only 4 segments >98.0% match found of total 8 segments in input file
Unassigned_PA
Not assigned: Only 5 segments >98.0% match found of total 8 segments in input file
Unassigned_PA
Not assigned: Only 2 segments >98.0% match found of total 8 segments in input file
Unassigned_PA
D1.1_PA
D1.3_PA
B3.13_PA
Not assigned: Only 0 segments >98.0% match found of total 8 segments in input file
Unassigned_PA
Not assigned: Only 3 segments >98.0% match found of total 8 segments in input file
Unassigned_PA
Not assigned: Only 6 segments >98.0% match found of total 8 segments in input file
Unassigned_PA
Not assigned: No Matching Genotypes
Unassigned_HA
Not assigned: Only 7 segments >98.0% match found of total 8 segments in input file
Unassigned_HA
Not assigned: Only 1 segments >98.0% m

In [51]:
# print(metadata_genoflu[metadata_genoflu["Genotype"] == "B3.2"])

In [53]:
# Create FASTA files

os.chdir(complete_files)

# names = []

for df in segment_genotype_dfs:
    print(df)
    df = df.reset_index()
    if len(df["Genotype"].values[0]) > 0:
        # print(df)
        if "Not assigned" not in df["Genotype"].values[0]:
            file_name = df["Genotype"].values[0] + "_" + df["Segment_Name"].values[0] + "_" + date_range + ".fasta"
            output_file = open(complete_files + file_name, "w")

            for index, row in df.iterrows():
                name = df.loc[index, "Name"]
                name = name.replace(" ", "_")
                # names.append(name)
                sequence = df.loc[index, "sequence"]
                # First is header, second is sequence
                output_file.write(name + "\n")
                output_file.write(sequence + "\n")
        else:
            file_name = "Unassigned_" + df["Segment_Name"].values[0] + "_" + date_range + ".fasta"
            output_file = open(complete_files + file_name, "w")

            for index, row in df.iterrows():
                name = df.loc[index, "Name"]
                name = name.replace(" ", "_")
                # names.append(name)
                sequence = df.loc[index, "sequence"]
                # First is header, second is sequence
                output_file.write(name + "\n")
                output_file.write(sequence + "\n")
    output_file.close()

        Accession                                      GenBank_Title  \
8      PX499386.1  Influenza A virus (A/Aves/Queretaro/CPA-19313-...   
856    PV949116.1  Influenza A virus (A/Duck/NY/24-035330-001-ori...   
864    PV949132.1  Influenza A virus (A/Duck/PA/24-034866-001-ori...   
872    PV949140.1  Influenza A virus (A/Duck/PA/24-035135-002-ori...   
880    PV949148.1  Influenza A virus (A/Duck/PA/24-035135-004-ori...   
...           ...                                                ...   
33824  PQ379372.1  Influenza A virus (A/cattle/MI/24-019382-007-o...   
34736  PQ197155.1  Influenza A virus (A/Turkey/MN/24-019077-001-o...   
35768  PQ109564.1  Influenza A virus (A/cattle/ID/24-016688-001-o...   
35792  PQ109604.1  Influenza A virus (A/cattle/SD/24-015059-001-o...   
35944  PQ051159.1  Influenza A virus (A/House Sparrow/NM/24-01478...   

                Host Collection_Date SRA_Accession  \
8               aves      2022-09-26   SRR36016324   
856             duck      2

In [ ]:
# # Concatenate with new Andersen sequences

# os.chdir(combined_files)

# # andersen = home + "Andersen/complete/" + date_range + "/"

# # Andersen files
# filenames_andersen = []
# for genotype in genotypes:
#     # print(gisaid_andersen + genotype.replace(".", "_") + "/")
#     # for dirpath, dirs, files in os.walk(gisaid_andersen + genotype.replace(".", "_") + "/" + date_range + "_" + genotype.replace(".", "_") + "/"):
#     for dirpath, dirs, files in os.walk(andersen): # Find the fasta file
#         for file in files:
#             file_name = os.path.join(dirpath, file) # Get file name
#             filenames_andersen.append(file_name)
#         break 

# # NCBI Virus files
# filenames_ncbi = []
# for dirpath, dirs, files in os.walk(complete_files): # Find the fasta file
#     for file in files:
#         file_name = os.path.join(dirpath, file) # Get file name
#         filenames_ncbi.append(file_name)
#     break 

# print(filenames_ncbi)

# common_genotypes = set()
# # Concatenate the two -- should not have any overlap due to dates and deduplication 
# for a_file in filenames_andersen:
#     partial_filename_a = a_file.split("_")[-4].split("/")[-1] + "_" + a_file.split("_")[-3]
#     for nv_file in filenames_ncbi:
#         partial_filename_nv = nv_file.split("_")[-3].split("/")[-1] + "_" + nv_file.split("_")[-2]
#         if partial_filename_a == partial_filename_nv:
#             common_genotypes.add(partial_filename_a)
#             filenames = [a_file, nv_file]
#             with open(combined_files + partial_filename_a + "_combined_" + date_range + ".fasta", 'w') as outfile:
#                 for fname in filenames:
#                     with open(fname) as infile:
#                         for line in infile:
#                             outfile.write(line)
#                         infile.close()
#                 outfile.close()

# print(common_genotypes)

# for a_file in filenames_andersen:
#     partial_filename_a = a_file.split("_")[-4].split("/")[-1] + "_" + a_file.split("_")[-3]
#     # If genotype not found in NCBI Virus, include it as well
#     if partial_filename_a not in common_genotypes:
#         with open(combined_files + partial_filename_a + "_combined_" + date_range + ".fasta", 'w') as outfile2:
#             with open(a_file) as infile2:
#                 for line in infile2:
#                     outfile2.write(line)
#                 infile2.close()
#             outfile2.close()